In [6]:
import numpy as np
file_path = 'C:/Users/shiek/OneDrive/Documents/Projects - DA/YoungDev Internship - 1 month/Expert Tasks/datasets/Dataset Task 1/Consumer_Shopping_Trends_2026 (6).csv'
data = np.genfromtxt(file_path, 
                     delimiter=',', 
                     names=True, 
                     dtype=None, 
                     encoding='utf-8')

column_names = data.dtype.names
numeric_cols = [name for name in column_names if data[name].dtype.kind in 'iuf'] # integer, unsigned, float
categorical_cols = [name for name in column_names if data[name].dtype.kind not in 'iuf']

numeric_matrix = np.column_stack([data[name] for name in numeric_cols])

print(f"Dataset Loaded Successfully!")
print(f"Total Rows: {len(data)}")
print(f"Numeric Columns found: {len(numeric_cols)}")
print(f"Categorical Columns found: {len(categorical_cols)}")
print("-" * 30)
print("Numeric Features:", numeric_cols)
print("Categorical Features:", categorical_cols)

Dataset Loaded Successfully!
Total Rows: 11789
Numeric Columns found: 22
Categorical Columns found: 3
------------------------------
Numeric Features: ['age', 'monthly_income', 'daily_internet_hours', 'smartphone_usage_years', 'social_media_hours', 'online_payment_trust_score', 'tech_savvy_score', 'monthly_online_orders', 'monthly_store_visits', 'avg_online_spend', 'avg_store_spend', 'discount_sensitivity', 'return_frequency', 'avg_delivery_days', 'delivery_fee_sensitivity', 'free_return_importance', 'product_availability_online', 'impulse_buying_score', 'need_touch_feel_score', 'brand_loyalty_score', 'environmental_awareness', 'time_pressure_level']
Categorical Features: ['gender', 'city_tier', 'shopping_preference']


Descriptive Profiling & Confidence Intervals.

In [7]:
import numpy as np

means = np.mean(numeric_matrix, axis=0)
stds = np.std(numeric_matrix, axis=0)
counts = numeric_matrix.shape[0]

print("---GLOBAL NUMERIC DESCRIPTORS ---")
for i, col in enumerate(numeric_cols):
    print(f"{col:30} | Mean: {means[i]:12.2f} | Std: {stds[i]:12.2f}")

print("\n" + "="*50 + "\n")

def calculate_ci_95(data_array):
    mean = np.mean(data_array)
    std = np.std(data_array)
    n = len(data_array)
    z = 1.96  # Z-score for 95% confidence
    
    margin_of_error = z * (std / np.sqrt(n))
    return mean - margin_of_error, mean + margin_of_error

spend_ci = calculate_ci_95(data['avg_online_spend'])
income_ci = calculate_ci_95(data['monthly_income'])

print("---KEY BUSINESS CONFIDENCE INTERVALS (95%) ---")
print(f"Average Online Spend: ${spend_ci[0]:.2f} to ${spend_ci[1]:.2f}")
print(f"Monthly Income:       ${income_ci[0]:.2f} to ${income_ci[1]:.2f}")

---GLOBAL NUMERIC DESCRIPTORS ---
age                            | Mean:        48.73 | Std:        17.90
monthly_income                 | Mean:    131704.28 | Std:     68117.84
daily_internet_hours           | Mean:         6.01 | Std:         1.98
smartphone_usage_years         | Mean:         7.60 | Std:         4.01
social_media_hours             | Mean:         2.51 | Std:         1.26
online_payment_trust_score     | Mean:         5.50 | Std:         2.88
tech_savvy_score               | Mean:         5.53 | Std:         2.89
monthly_online_orders          | Mean:        24.68 | Std:        14.43
monthly_store_visits           | Mean:         9.48 | Std:         5.73
avg_online_spend               | Mean:     74554.93 | Std:     43165.30
avg_store_spend                | Mean:     75661.63 | Std:     43413.95
discount_sensitivity           | Mean:         5.50 | Std:         2.88
return_frequency               | Mean:         4.47 | Std:         2.89
avg_delivery_days             

In [8]:
import numpy as np

full_corr_matrix = np.corrcoef(numeric_matrix, rowvar=False)

def get_correlation(col1_name, col2_name):
    idx1 = numeric_cols.index(col1_name)
    idx2 = numeric_cols.index(col2_name)
    return full_corr_matrix[idx1, idx2]

tech_vs_spend = get_correlation('tech_savvy_score', 'avg_online_spend')
internet_vs_impulse = get_correlation('daily_internet_hours', 'impulse_buying_score')
fee_vs_orders = get_correlation('delivery_fee_sensitivity', 'monthly_online_orders')

print("---KEY BEHAVIORAL CORRELATIONS ---")
print(f"Tech-Savviness vs. Online Spend:          {tech_vs_spend:10.4f}")
print(f"Internet Hours vs. Impulse Buying:       {internet_vs_impulse:10.4f}")
print(f"Delivery Fee Sensitivity vs. Order Volume: {fee_vs_orders:10.4f}")

print("\n" + "="*50 + "\n")

spend_idx = numeric_cols.index('avg_online_spend')
spend_correlations = full_corr_matrix[spend_idx]

sorted_indices = np.argsort(np.abs(spend_correlations))[::-1]

print("---TOP 5 DRIVERS OF ONLINE SPENDING ---")
count = 0
for idx in sorted_indices:
    if numeric_cols[idx] == 'avg_online_spend': continue
    if count >= 5: break
    print(f"{count+1}. {numeric_cols[idx]:30} | Correlation: {spend_correlations[idx]:.4f}")
    count += 1

print("\n" + "="*50)
print("STRATEGIC INSIGHT:")
if abs(tech_vs_spend) < 0.1:
    print("The correlation between Tech-Savviness and Spending is very low.")
    print("Insight: Our platform is accessible to everyone. We don't need to limit")
    print("marketing to 'techies'; we can broad-target to increase market share.")
else:
    print(f"There is a visible correlation of {tech_vs_spend:.2f} for tech-savvy users.")
    print("Insight: Focus digital advertising on high-tech platforms (Reddit, Tech Blogs).")

---KEY BEHAVIORAL CORRELATIONS ---
Tech-Savviness vs. Online Spend:             -0.0053
Internet Hours vs. Impulse Buying:          -0.0003
Delivery Fee Sensitivity vs. Order Volume:     0.0100


---TOP 5 DRIVERS OF ONLINE SPENDING ---
1. age                            | Correlation: 0.0170
2. monthly_income                 | Correlation: 0.0137
3. discount_sensitivity           | Correlation: 0.0129
4. return_frequency               | Correlation: 0.0097
5. time_pressure_level            | Correlation: -0.0090

STRATEGIC INSIGHT:
The correlation between Tech-Savviness and Spending is very low.
Insight: Our platform is accessible to everyone. We don't need to limit
marketing to 'techies'; we can broad-target to increase market share.


In [10]:
import numpy as np

print("--- T-TEST: Tier 1 vs Tier 3 Spending ---")
t1_data = data['avg_online_spend'][data['city_tier'] == 'Tier 1']
t3_data = data['avg_online_spend'][data['city_tier'] == 'Tier 3']

m1, m2 = np.mean(t1_data), np.mean(t3_data)
v1, v2 = np.var(t1_data, ddof=1), np.var(t3_data, ddof=1)
n1, n2 = len(t1_data), len(t3_data)

pooled_se = np.sqrt(v1/n1 + v2/n2)
t_stat = (m1 - m2) / pooled_se

print(f"Tier 1 Mean: ${m1:.2f} | Tier 3 Mean: ${m2:.2f}")
print(f"Calculated T-Statistic: {t_stat:.4f}")
print(f"Degrees of Freedom: {n1 + n2 - 2}")

if abs(t_stat) > 1.96:
    print("RESULT: The difference is Statistically Significant (|t| > 1.96).")
else:
    print("RESULT: Not Significant. The difference is likely due to random noise.")

print("\n" + "="*50 + "\n")

print("---MANUAL ANOVA: Spending by Shopping Preference ---")

unique_prefs = np.unique(data['shopping_preference'])
groups = [data['avg_online_spend'][data['shopping_preference'] == p] for p in unique_prefs]

all_spend = data['avg_online_spend']
grand_mean = np.mean(all_spend)

ss_between = sum(len(g) * (np.mean(g) - grand_mean)**2 for g in groups)
ss_within = sum(np.sum((g - np.mean(g))**2) for g in groups)

df_between = len(unique_prefs) - 1
df_within = len(all_spend) - len(unique_prefs)

ms_between = ss_between / df_between
ms_within = ss_within / df_within
f_stat = ms_between / ms_within

print(f"Groups Analyzed: {unique_prefs}")
print(f"F-Statistic: {f_stat:.4f}")
print(f"DF (Between, Within): ({df_between}, {df_within})")

if f_stat > 3.0:
    print("RESULT: Significant variation exists between shopping preferences!")
else:
    print("RESULT: No significant variation between groups.")

--- T-TEST: Tier 1 vs Tier 3 Spending ---
Tier 1 Mean: $74520.52 | Tier 3 Mean: $74244.91
Calculated T-Statistic: 0.2839
Degrees of Freedom: 7929
RESULT: Not Significant. The difference is likely due to random noise.


---MANUAL ANOVA: Spending by Shopping Preference ---
Groups Analyzed: ['Hybrid' 'Online' 'Store']
F-Statistic: 1.2313
DF (Between, Within): (2, 11786)
RESULT: No significant variation between groups.


In [13]:
import numpy as np

def perform_manual_regression(x, y, x_label, y_label):
    n = len(x)
    sum_x = np.sum(x)
    sum_y = np.sum(y)
    sum_xx = np.sum(x**2)
    sum_xy = np.sum(x * y)
    numerator = (n * sum_xy) - (sum_x * sum_y)
    denominator = (n * sum_xx) - (sum_x**2)
    m = numerator / denominator
    c = (sum_y - (m * sum_x)) / n
    y_pred = m * x + c
    ss_res = np.sum((y - y_pred)**2) # Sum of Squares Residuals
    ss_tot = np.sum((y - np.mean(y))**2) # Total Sum of Squares
    r_squared = 1 - (ss_res / ss_tot)
    
    return m, c, r_squared

income = data['monthly_income']
spend = data['avg_online_spend']
m1, c1, r1 = perform_manual_regression(income, spend, "Income", "Spend")

soc_media = data['social_media_hours']
m2, c2, r2 = perform_manual_regression(soc_media, spend, "Social Media", "Spend")

print("--- REGRESSION MODEL A: Income -> Online Spend ---")
print(f"Equation: Spend = ({m1:.4f} * Income) + {c1:.2f}")
print(f"R-squared: {r1:.6f}")
print(f"Insight: For every $1,000 increase in income, spend changes by ${m1*1000:.2f}")

print("\n" + "-"*50 + "\n")

print("--- REGRESSION MODEL B: Social Media -> Online Spend ---")
print(f"Equation: Spend = ({m2:.4f} * Hours) + {c2:.2f}")
print(f"R-squared: {r2:.6f}")
print(f"Insight: Every extra hour on social media changes spend by ${m2:.2f}")

print("\n" + "="*50)
print("STRATEGIC INSIGHT:")
if r1 < 0.01 and r2 < 0.01:
    print("WARNING: Both R-squared values are extremely low.")
    print("This means neither Income nor Social Media hours are strong predictors")
    print("of total spending. Suggestion: We should stop focusing on 'Wealthy'")
    print("targets and instead analyze 'Discount Sensitivity' or 'Brand Loyalty'")
    print("to find the real drivers of our revenue.")

--- REGRESSION MODEL A: Income -> Online Spend ---
Equation: Spend = (0.0087 * Income) + 73410.54
R-squared: 0.000188
Insight: For every $1,000 increase in income, spend changes by $8.69

--------------------------------------------------

--- REGRESSION MODEL B: Social Media -> Online Spend ---
Equation: Spend = (-57.4621 * Hours) + 74699.42
R-squared: 0.000003
Insight: Every extra hour on social media changes spend by $-57.46

STRATEGIC INSIGHT:
This means neither Income nor Social Media hours are strong predictors
of total spending. Suggestion: We should stop focusing on 'Wealthy'
targets and instead analyze 'Discount Sensitivity' or 'Brand Loyalty'
to find the real drivers of our revenue.


In [14]:
import numpy as np

def perform_manual_chi_square(cat_var1, cat_var2, label1, label2):
    levels1 = np.unique(cat_var1)
    levels2 = np.unique(cat_var2)
    observed = np.zeros((len(levels1), len(levels2)))
    for i, val1 in enumerate(levels1):
        for j, val2 in enumerate(levels2):
            observed[i, j] = np.sum((cat_var1 == val1) & (cat_var2 == val2))
    row_totals = np.sum(observed, axis=1)
    col_totals = np.sum(observed, axis=0)
    grand_total = np.sum(observed)
    expected = np.outer(row_totals, col_totals) / grand_total
    chi_sq_stat = np.sum((observed - expected)**2 / expected)
    df = (len(levels1) - 1) * (len(levels2) - 1)
    return observed, chi_sq_stat, df, levels1, levels2
obs_table, chi_stat, df_val, row_labs, col_labs = perform_manual_chi_square(
    data['gender'], 
    data['shopping_preference'],
    "Gender",
    "Preference"
)

print("--- CHI-SQUARE TEST: Gender vs. Shopping Preference ---")
print("Observed Contingency Table:")
print(f"{'':10}", end="")
for label in col_labs:
    print(f"{label:12}", end="")
print()

for i, row_label in enumerate(row_labs):
    print(f"{row_label:10}", end="")
    for val in obs_table[i]:
        print(f"{int(val):12d}", end="")
    print()

print("\n" + "-"*50)
print(f"Chi-Square Statistic: {chi_stat:.4f}")
print(f"Degrees of Freedom:    {df_val}")

print("\n" + "="*50)
print("STRATEGIC INSIGHT:")

if chi_stat > 9.49:
    print("RESULT: Statistically Significant Relationship Found.")
    print("Insight: Shopping preference is NOT independent of gender. Certain")
    print("genders lean significantly toward specific shopping modes (e.g., Store vs Online).")
    print("Action: Optimize the user journey specifically for the preferred channel of each segment.")
else:
    print("RESULT: No Significant Relationship Found.")
    print("Insight: Shopping preferences are distributed evenly across genders.")
    print("Action: A unified cross-channel strategy will work effectively for all customers.")

--- CHI-SQUARE TEST: Gender vs. Shopping Preference ---
Observed Contingency Table:
          Hybrid      Online      Store       
Female             145         408        3378
Male               102         378        3486
Other              122         390        3380

--------------------------------------------------
Chi-Square Statistic: 10.1805
Degrees of Freedom:    4

STRATEGIC INSIGHT:
RESULT: Statistically Significant Relationship Found.
Insight: Shopping preference is NOT independent of gender. Certain
genders lean significantly toward specific shopping modes (e.g., Store vs Online).
Action: Optimize the user journey specifically for the preferred channel of each segment.


In [15]:
import numpy as np
spend_data = data['avg_online_spend']
income_data = data['monthly_income']
internet_data = data['daily_internet_hours']

hvc_threshold = np.percentile(spend_data, 90)
hvc_mask = spend_data >= hvc_threshold
hvc_avg_income = np.mean(income_data[hvc_mask])
gen_avg_income = np.mean(income_data[~hvc_mask])
hvc_avg_internet = np.mean(internet_data[hvc_mask])
gen_avg_internet = np.mean(internet_data[~hvc_mask])
wallet_share = (spend_data / income_data) * 100
avg_wallet_share = np.mean(wallet_share)

q1 = np.percentile(spend_data, 25)
q3 = np.percentile(spend_data, 75)
iqr = q3 - q1
upper_bound = q3 + (1.5 * iqr)
lower_bound = q1 - (1.5 * iqr)
outliers = spend_data[spend_data > upper_bound]
outlier_count = len(outliers)

print("---HIGH-VALUE CUSTOMER (HVC) PROFILING ---")
print(f"90th Percentile Spend Threshold: ${hvc_threshold:.2f}")
print(f"Average Income (HVCs): ${hvc_avg_income:.2f}")
print(f"Average Income (Others): ${gen_avg_income:.2f}")
print(f"Daily Internet Hours (HVCs): {hvc_avg_internet:.2f} hrs")
print(f"Daily Internet Hours (Others): {gen_avg_internet:.2f} hrs")
print("\n" + "-"*50 + "\n")

print("---WALLET SHARE ANALYSIS ---")
print(f"Average Online Wallet Share: {avg_wallet_share:.2f}%")
print("\n" + "-"*50 + "\n")

print("---REVENUE STABILITY (OUTLIER DETECTION) ---")
print(f"Q1 (25th Percentile): ${q1:.2f}")
print(f"Q3 (75th Percentile): ${q3:.2f}")
print(f"IQR: ${iqr:.2f}")
print(f"Upper Outlier Bound: ${upper_bound:.2f}")
print(f"Number of Upper Outliers: {outlier_count}")
print("\n" + "="*50)

print("STRATEGIC INSIGHT:")
print("1. Customer Profile: HVCs are not necessarily the highest earners. Their")
print(f"average income (${hvc_avg_income:.0f}) is very close to the general average.")
print("Insight: Focus on engagement and loyalty, not just high-income leads.")
print(f"2. Wallet Share: At {avg_wallet_share:.1f}%, we have high penetration. Growth")
print("must come from increasing total income or acquisition, rather than upselling.")
print(f"3. Stability: With {outlier_count} outliers, our revenue is predictable.")
print("The business is not overly dependent on a few 'super-spenders.'")

---HIGH-VALUE CUSTOMER (HVC) PROFILING ---
90th Percentile Spend Threshold: $134594.40
Average Income (HVCs): $132857.13
Average Income (Others): $131576.18
Daily Internet Hours (HVCs): 6.06 hrs
Daily Internet Hours (Others): 6.01 hrs

--------------------------------------------------

---WALLET SHARE ANALYSIS ---
Average Online Wallet Share: 89.50%

--------------------------------------------------

---REVENUE STABILITY (OUTLIER DETECTION) ---
Q1 (25th Percentile): $36797.00
Q3 (75th Percentile): $112134.00
IQR: $75337.00
Upper Outlier Bound: $225139.50
Number of Upper Outliers: 0

STRATEGIC INSIGHT:
1. Customer Profile: HVCs are not necessarily the highest earners. Their
average income ($132857) is very close to the general average.
Insight: Focus on engagement and loyalty, not just high-income leads.
2. Wallet Share: At 89.5%, we have high penetration. Growth
must come from increasing total income or acquisition, rather than upselling.
3. Stability: With 0 outliers, our revenue is

In [17]:
import numpy as np
features = ['age', 'monthly_income', 'brand_loyalty_score', 'environmental_awareness']
X = np.column_stack([data[f] for f in features])
X_min = X.min(axis=0)
X_max = X.max(axis=0)
X_scaled = (X - X_min) / (X_max - X_min)
k = 3
np.random.seed(42)
centroids = X_scaled[np.random.choice(X_scaled.shape[0], k, replace=False)]

for _ in range(10):
    distances = np.linalg.norm(X_scaled[:, np.newaxis] - centroids, axis=2)
    labels = np.argmin(distances, axis=1)
    new_centroids = np.array([X_scaled[labels == i].mean(axis=0) for i in range(k)])
    if np.all(centroids == new_centroids): break
    centroids = new_centroids
    
print("---CUSTOMER SEGMENTATION (K-MEANS) ---")
for i in range(k):
    cluster_size = np.sum(labels == i)
    cluster_actuals = X[labels == i].mean(axis=0)
    print(f"Cluster {i+1} (Size: {cluster_size}):")
    print(f"  Avg Age: {cluster_actuals[0]:.1f} | Avg Income: ${cluster_actuals[1]:.0f}")
    print(f"  Loyalty: {cluster_actuals[2]:.1f}/10 | Eco-Aware: {cluster_actuals[3]:.1f}/10")
    
print("\n" + "="*50)
print("STRATEGIC INSIGHT:")
print("We have successfully identified 3 distinct customer personas:")
print("1. Target Cluster: Look for the group with high Brand Loyalty and high")
print("   Environmental Awareness for our upcoming 'Green' product line.")
print("2. Efficiency: By segmenting, we can stop sending generic emails and")
print("   start sending personalized offers that resonate with each group's values.")

---CUSTOMER SEGMENTATION (K-MEANS) ---
Cluster 1 (Size: 3771):
  Avg Age: 50.9 | Avg Income: $128825
  Loyalty: 3.3/10 | Eco-Aware: 3.2/10
Cluster 2 (Size: 4246):
  Avg Age: 54.6 | Avg Income: $127958
  Loyalty: 5.2/10 | Eco-Aware: 8.3/10
Cluster 3 (Size: 3772):
  Avg Age: 40.0 | Avg Income: $138800
  Loyalty: 8.2/10 | Eco-Aware: 4.5/10

STRATEGIC INSIGHT:
We have successfully identified 3 distinct customer personas:
1. Target Cluster: Look for the group with high Brand Loyalty and high
   Environmental Awareness for our upcoming 'Green' product line.
2. Efficiency: By segmenting, we can stop sending generic emails and
   start sending personalized offers that resonate with each group's values.


In [18]:
import numpy as np

tech_scores = data['tech_savvy_score']
spend_data = data['avg_online_spend']

median_tech = np.median(tech_scores)
high_tech_mask = tech_scores >= median_tech
low_tech_mask = tech_scores < median_tech
mean_spend_high = np.mean(spend_data[high_tech_mask])
mean_spend_low = np.mean(spend_data[low_tech_mask])
count_high = np.sum(high_tech_mask)
count_low = np.sum(low_tech_mask)

print("---THE TECH-SAVVY PARADOX ---")
print(f"Median Tech-Savvy Score: {median_tech}")
print(f"High Tech-Savvy (Score >= {median_tech}): {count_high} users | Avg Spend: ${mean_spend_high:.2f}")
print(f"Low Tech-Savvy (Score < {median_tech}): {count_low} users | Avg Spend: ${mean_spend_low:.2f}")
print(f"Spending Difference: ${abs(mean_spend_high - mean_spend_low):.2f}")
print("\n" + "="*50)

print("STRATEGIC INSIGHT:")
if abs(mean_spend_high - mean_spend_low) < (0.05 * mean_spend_high):
    print("THE PARADOX: Tech-savviness is NOT a barrier to spending.")
    print("Our data shows that users with lower technical scores spend almost")
    print("exactly the same as power users. This is a testament to the UX design.")
    print("Strategy: We should aggressively target older demographics or less")
    print("digitally-native platforms, as they are just as profitable as techies.")
else:
    print("Finding: Technical proficiency does impact spending behavior.")
    print("Strategy: Implement 'Simplified Mode' or guided tutorials for the")
    print("lower-spending tech group to reduce friction and increase conversion.")

---THE TECH-SAVVY PARADOX ---
Median Tech-Savvy Score: 6.0
High Tech-Savvy (Score >= 6.0): 5924 users | Avg Spend: $74135.50
Low Tech-Savvy (Score < 6.0): 5865 users | Avg Spend: $74978.57
Spending Difference: $843.07

STRATEGIC INSIGHT:
THE PARADOX: Tech-savviness is NOT a barrier to spending.
Our data shows that users with lower technical scores spend almost
exactly the same as power users. This is a testament to the UX design.
Strategy: We should aggressively target older demographics or less
digitally-native platforms, as they are just as profitable as techies.
